# 47. Multimodal models and contrastive representations

**Level:** Advanced

**Prerequisites:** 34-36: embeddings and attention.

[Course index](00_START_HERE.ipynb) | [Previous](46_Generative_Models_Autoencoders_and_Diffusion.ipynb) | [Next](48_Pydantic_FastAPI_and_Model_Serving.ipynb)

**Topics:** Multimodal AI, Vision-language models, Audio, Video, Contrastive learning, Shared embeddings.

**Matching script:** `examples\47_Multimodal_Models_and_Contrastive_Learning.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

Multimodal systems combine information such as text, images, audio, and video. They may encode
each modality separately then align representations, project image/audio features into a
language model, or train an integrated architecture. A shared interface does not imply identical
accuracy or output support across modalities.

Contrastive training pulls paired representations together and separates mismatches according
to a loss. Paired image-text embeddings can support cross-modal retrieval without generating
a caption. Captioning, speech recognition, speech generation, video understanding, and image
synthesis are distinct tasks with different evaluation requirements.

Inspect modality preprocessing, resolution, duration, timestamps, alignment, and token costs.
A model can hallucinate visual details or miss temporal events. Avoid biometric or personal
data without an appropriate lawful and reviewed process.

The example optimizes tiny paired embedding tables from scratch. 'Image' and 'text' are names
for synthetic paired IDs, not a claim to encode real pictures or understand language.

## Mathematics or data contract

A paired contrastive objective can average cross_entropy(similarity, correct_pair) in both directions.

## Important limitation

Synthetic pair alignment teaches the objective only; it is not a pretrained multimodal model.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Learn alignment between two synthetic modalities

**Purpose:** Train and verify the matching objective directly.

### Read the code in this order
1. Create independent trainable embedding tables for paired IDs.
2. Normalize both before similarity.
3. Train a symmetric contrastive loss.
4. Assert diagonal retrieval after optimization.

### Syntax and behavior to notice
- `backward()` computes/accumulates gradients through the recorded computation graph.
- `zero_grad()` clears earlier gradients before the next independent training update.
- `optimizer.step()` updates only the parameters owned by that optimizer.
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
synthetic matched pair IDs: [0, 1, 2, 3]
contrastive loss: 0.0005
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)
image_ids = nn.Parameter(torch.randn(4, 8))
text_ids = nn.Parameter(torch.randn(4, 8))
optimizer = torch.optim.Adam([image_ids, text_ids], lr=0.05)
labels = torch.arange(4)
for _ in range(100):
    optimizer.zero_grad()
    similarities = F.normalize(image_ids, dim=1) @ F.normalize(text_ids, dim=1).T / 0.1
    loss = (F.cross_entropy(similarities, labels) + F.cross_entropy(similarities.T, labels)) / 2
    loss.backward()
    optimizer.step()
assert similarities.argmax(1).tolist() == [0, 1, 2, 3]
print("synthetic matched pair IDs:", similarities.argmax(1).tolist())
print("contrastive loss:", round(loss.item(), 4))

synthetic matched pair IDs: [0, 1, 2, 3]
contrastive loss: 0.0005


### Interpret example 1

The result should demonstrate this purpose: **Train and verify the matching objective directly.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Introduce duplicate concepts and discuss why treating every other batch item as a negative may be wrong.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Multimodal AI` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: A paired contrastive objective can average cross_entropy(similarity, correct_pair) in both directions.
- Name the boundary of the demonstration: Synthetic pair alignment teaches the objective only; it is not a pretrained multimodal model.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://arxiv.org/abs/2103.00020`
- `https://docs.pytorch.org/tutorials/beginner/basics/intro.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.